> **Repository version.** This notebook was cleaned from the frozen manuscript workflow:
> outputs were removed and local absolute paths were replaced by the repository `PROJECT_DIR`.
> The statistical logic was not intentionally changed.


In [ ]:
# Repository path setup
from pathlib import Path
import os

_here = Path.cwd().resolve()
REPO_ROOT = _here.parent if _here.name == "notebooks" else _here
PROJECT_DIR = Path(
    os.environ.get("CGN_PROJECT_DIR", REPO_ROOT / "workspace")
).expanduser().resolve()

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "reproduction" / "results").mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Working data/results directory:", PROJECT_DIR)


In [ ]:
# ============================================================
# ORTHOGONAL SPATIAL VALIDATION
#
# MAC → FIB
#
# Metric:
# permutation-corrected nearest-FIB distance from MAC cells
#
# Positive proximity_z =
# FIB cells are closer to MAC than expected by chance
# ============================================================

from pathlib import Path

import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from scipy.spatial import cKDTree
from patsy import bs


# ============================================================
# 0. Paths
# ============================================================

base = Path(str(PROJECT_DIR))

figdir = base / "figures"

figdir.mkdir(
    parents=True,
    exist_ok=True
)


cell_meta_path = (
    base /
    "roi782_cell_metadata.pkl"
)

roi_path = (
    base /
    "roi_782_PC1_primary.h5ad"
)


# ============================================================
# 1. Load lightweight data
# ============================================================

cells = pd.read_pickle(
    cell_meta_path
)

roi_ad = ad.read_h5ad(
    roi_path
)

print(
    "Cell metadata:",
    cells.shape
)

print(
    "ROI object:",
    roi_ad.shape
)


# ============================================================
# 2. Define LN / anti-GBM common PC1 support
# ============================================================

meta = roi_ad.obs[
    roi_ad.obs["Disease"].isin(
        ["SLE", "GBM"]
    )
].copy()


ranges = (
    meta.groupby(
        "Disease",
        observed=True
    )["PC1_crescent"]
    .agg(
        ["min", "max"]
    )
)


pc_low = (
    ranges["min"].max()
)

pc_high = (
    ranges["max"].min()
)


meta = meta[
    meta["PC1_crescent"].between(
        pc_low,
        pc_high
    )
].copy()


common_ids = set(
    meta.index.astype(str)
)


print(
    "\nCommon PC1:",
    pc_low,
    "to",
    pc_high
)

print(
    "\nROI counts:"
)

print(
    meta["Disease"].value_counts()
)

print(
    "\nPatient counts:"
)

print(
    meta.groupby(
        "Disease",
        observed=True
    )["Patient_Sample_ID"].nunique()
)


# ============================================================
# 3. Keep only these ROI cells
# ============================================================

cells_use = cells[
    cells["roi_id"]
    .astype(str)
    .isin(
        common_ids
    )
].copy()


print(
    "\nCells used:",
    len(cells_use)
)


# ============================================================
# 4. Permutation nearest-distance function
#
# We condition on:
# - observed MAC positions
# - number of FIB cells in that ROI
#
# FIB labels are randomly assigned among NON-MAC cells.
# ============================================================

def mac_fib_proximity_one_roi(
    d,
    n_perm=100,
    min_mac=3,
    min_fib=3,
    seed=2026
):

    labels = (
        d["celltype_l1"]
        .astype(str)
        .to_numpy()
    )


    mac_mask = (
        labels == "MAC"
    )

    fib_mask = (
        labels == "FIB"
    )


    n_mac = int(
        mac_mask.sum()
    )

    n_fib = int(
        fib_mask.sum()
    )


    if (
        n_mac < min_mac
        or
        n_fib < min_fib
    ):

        return {
            "observed_median_distance":
                np.nan,

            "null_mean_distance":
                np.nan,

            "null_sd_distance":
                np.nan,

            "proximity_z":
                np.nan,

            "n_MAC":
                n_mac,

            "n_FIB":
                n_fib
        }


    xy = d[
        [
            "x_centroid",
            "y_centroid"
        ]
    ].to_numpy(
        dtype=float
    )


    mac_xy = xy[
        mac_mask
    ]


    fib_xy = xy[
        fib_mask
    ]


    # --------------------------------
    # Observed distance
    # --------------------------------

    fib_tree = cKDTree(
        fib_xy
    )


    observed_distances, _ = (
        fib_tree.query(
            mac_xy,
            k=1
        )
    )


    observed_median = float(
        np.median(
            observed_distances
        )
    )


    # --------------------------------
    # Null candidate locations:
    # all cells that are not MAC
    # --------------------------------

    candidate_xy = xy[
        ~mac_mask
    ]


    n_candidates = len(
        candidate_xy
    )


    if n_fib > n_candidates:

        return {
            "observed_median_distance":
                np.nan,

            "null_mean_distance":
                np.nan,

            "null_sd_distance":
                np.nan,

            "proximity_z":
                np.nan,

            "n_MAC":
                n_mac,

            "n_FIB":
                n_fib
        }


    # ROI-specific deterministic seed
    rng = np.random.default_rng(
        seed
        +
        len(d)
        +
        n_mac * 17
        +
        n_fib * 31
    )


    null_medians = np.empty(
        n_perm,
        dtype=float
    )


    for i in range(
        n_perm
    ):

        idx = rng.choice(
            n_candidates,
            size=n_fib,
            replace=False
        )


        random_fib_xy = (
            candidate_xy[
                idx
            ]
        )


        tree = cKDTree(
            random_fib_xy
        )


        null_dist, _ = (
            tree.query(
                mac_xy,
                k=1
            )
        )


        null_medians[i] = (
            np.median(
                null_dist
            )
        )


    null_mean = float(
        np.mean(
            null_medians
        )
    )


    null_sd = float(
        np.std(
            null_medians,
            ddof=1
        )
    )


    if (
        not np.isfinite(
            null_sd
        )
        or null_sd == 0
    ):

        proximity_z = np.nan

    else:

        # Positive =
        # observed MAC-FIB distance
        # is SMALLER than random expectation

        proximity_z = (
            null_mean
            -
            observed_median
        ) / null_sd


    return {
        "observed_median_distance":
            observed_median,

        "null_mean_distance":
            null_mean,

        "null_sd_distance":
            null_sd,

        "proximity_z":
            proximity_z,

        "n_MAC":
            n_mac,

        "n_FIB":
            n_fib
    }


# ============================================================
# 5. Calculate ROI-level proximity
# ============================================================

rows = []


roi_groups = list(
    cells_use.groupby(
        "roi_id",
        observed=True
    )
)


n_total = len(
    roi_groups
)


for i, (
    roi_id,
    d
) in enumerate(
    roi_groups,
    start=1
):


    result = (
        mac_fib_proximity_one_roi(
            d,
            n_perm=100,
            min_mac=3,
            min_fib=3
        )
    )


    result[
        "roi_id"
    ] = str(
        roi_id
    )


    rows.append(
        result
    )


    if (
        i % 25 == 0
        or i == n_total
    ):

        print(
            f"Processed "
            f"{i} / {n_total} ROIs"
        )


distance_df = pd.DataFrame(
    rows
).set_index(
    "roi_id"
)


# ============================================================
# 6. Add metadata
# ============================================================

distance_df[
    "Disease"
] = (
    roi_ad.obs.loc[
        distance_df.index,
        "Disease"
    ]
    .astype(str)
)


distance_df[
    "Patient"
] = (
    roi_ad.obs.loc[
        distance_df.index,
        "Patient_Sample_ID"
    ]
    .astype(str)
)


distance_df[
    "PC1"
] = (
    roi_ad.obs.loc[
        distance_df.index,
        "PC1_crescent"
    ]
)


# ============================================================
# 7. Coverage check
# ============================================================

valid = distance_df.dropna(
    subset=[
        "proximity_z"
    ]
).copy()


print(
    "\n=============================="
)

print(
    "VALID ROIs"
)

print(
    "=============================="
)


print(
    valid.groupby(
        "Disease",
        observed=True
    )[
        "proximity_z"
    ].count()
)


print(
    "\nPatients:"
)


print(
    valid.groupby(
        "Disease",
        observed=True
    )[
        "Patient"
    ].nunique()
)


print(
    "\nProximity distribution:"
)


display(
    valid[
        "proximity_z"
    ].describe()
)


# ============================================================
# 8. Formal Disease × PC1 interaction model
# ============================================================

d = valid[
    valid["Disease"].isin(
        ["SLE", "GBM"]
    )
].copy()


d["Disease"] = pd.Categorical(
    d["Disease"].astype(str),
    categories=[
        "SLE",
        "GBM"
    ]
)


# Patient-balanced weighting
n_roi = (
    d.groupby(
        "Patient",
        observed=True
    )["Patient"]
    .transform(
        "size"
    )
)


d[
    "patient_weight"
] = (
    1.0 /
    n_roi
)


fit = smf.wls(
    (
        "proximity_z ~ "
        "bs(PC1, df=3, degree=3, "
        "include_intercept=False) "
        "* C(Disease)"
    ),
    data=d,
    weights=d[
        "patient_weight"
    ]
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups":
            d["Patient"]
    }
)


terms = [
    term
    for term
    in fit.params.index
    if ":" in term
    and
    "C(Disease)"
    in term
    and
    "bs(PC1"
    in term
]


R = np.zeros(
    (
        len(terms),
        len(
            fit.params
        )
    )
)


for i, term in enumerate(
    terms
):

    R[
        i,
        fit.params
        .index
        .get_loc(
            term
        )
    ] = 1


wt = fit.wald_test(
    R,
    scalar=True
)


interaction_p = float(
    wt.pvalue
)


matrix_rank = (
    np.linalg.matrix_rank(
        fit.model.exog
    )
)


n_columns = (
    fit.model.exog.shape[
        1
    ]
)


print(
    "\n=============================="
)

print(
    "ORTHOGONAL MAC-FIB RESULT"
)

print(
    "=============================="
)


print(
    "Disease × PC1 P =",
    interaction_p
)


print(
    "Matrix rank =",
    matrix_rank
)


print(
    "Model columns =",
    n_columns
)


print(
    "ROIs =",
    len(d)
)


print(
    "Patients =",
    d["Patient"].nunique()
)


print(
    "GBM patients =",
    d.loc[
        d["Disease"]
        .astype(str)
        == "GBM",
        "Patient"
    ].nunique()
)


# ============================================================
# 9. Formal prediction curves
# ============================================================

grid = np.linspace(
    d["PC1"].min(),
    d["PC1"].max(),
    180
)


pred_list = []


for disease in [
    "SLE",
    "GBM"
]:

    newdata = pd.DataFrame({
        "PC1":
            grid,

        "Disease":
            pd.Categorical(
                [disease]
                *
                len(grid),

                categories=[
                    "SLE",
                    "GBM"
                ]
            )
    })


    pred = (
        fit.get_prediction(
            newdata
        )
        .summary_frame(
            alpha=0.05
        )
    )


    temp = pd.DataFrame({
        "PC1":
            grid,

        "Disease":
            disease,

        "fit":
            pred[
                "mean"
            ].to_numpy(),

        "lower":
            pred[
                "mean_ci_lower"
            ].to_numpy(),

        "upper":
            pred[
                "mean_ci_upper"
            ].to_numpy()
    })


    pred_list.append(
        temp
    )


pred_df = pd.concat(
    pred_list,
    ignore_index=True
)


# ============================================================
# 10. Screening figure
# ============================================================

palette = {
    "SLE":
        "#E69F00",

    "GBM":
        "#009E73"
}


display_name = {
    "SLE":
        "LN",

    "GBM":
        "anti-GBM"
}


fig, ax = plt.subplots(
    figsize=(
        7,
        5.5
    )
)


for disease in [
    "SLE",
    "GBM"
]:

    raw = d[
        d["Disease"]
        .astype(str)
        == disease
    ]


    pred = pred_df[
        pred_df["Disease"]
        == disease
    ]


    color = palette[
        disease
    ]


    ax.scatter(
        raw["PC1"],
        raw["proximity_z"],
        s=18,
        alpha=0.22,
        color=color
    )


    ax.plot(
        pred["PC1"],
        pred["fit"],
        linewidth=2.6,
        color=color,
        label=display_name[
            disease
        ]
    )


    ax.fill_between(
        pred["PC1"],
        pred["lower"],
        pred["upper"],
        color=color,
        alpha=0.14
    )


ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)


ax.set_xlabel(
    "Crescent progression (PC1)"
)


ax.set_ylabel(
    "MAC–FIB proximity z-score"
)


ax.set_title(
    (
        "Orthogonal MAC–FIB spatial validation\n"
        f"LN vs anti-GBM interaction "
        f"P = {interaction_p:.3g}"
    ),
    loc="left"
)


ax.legend(
    frameon=False
)


ax.spines[
    "top"
].set_visible(
    False
)


ax.spines[
    "right"
].set_visible(
    False
)


plt.tight_layout()


png_path = (
    figdir /
    "Figure4_orthogonal_MAC_FIB_distance_validation.png"
)


pdf_path = (
    figdir /
    "Figure4_orthogonal_MAC_FIB_distance_validation.pdf"
)


plt.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.savefig(
    pdf_path,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.show()


# ============================================================
# 11. Save
# ============================================================

distance_df.to_csv(
    base /
    "orthogonal_MAC_FIB_distance_data.csv"
)


result_df = pd.DataFrame({
    "metric": [
        "permutation_corrected_MAC_FIB_nearest_distance"
    ],

    "interaction_pvalue": [
        interaction_p
    ],

    "matrix_rank": [
        matrix_rank
    ],

    "n_columns": [
        n_columns
    ],

    "n_ROI": [
        len(d)
    ],

    "n_patients": [
        d["Patient"].nunique()
    ],

    "n_GBM_patients": [
        d.loc[
            d["Disease"]
            .astype(str)
            == "GBM",
            "Patient"
        ].nunique()
    ]
})


result_df.to_csv(
    base /
    "orthogonal_MAC_FIB_distance_result.csv",
    index=False
)


print(
    "\nSaved."
)